In [4]:
import warnings
import pandas as pd
# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    
    # Filter the subsequent prices correctly (inclusive of the entry_datetime)
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry_by_percentage(price_data, signal_datetime, percentage_change, side):
    signal_open_price = price_data.at[signal_datetime, 'Open']
    percentage_change_price = signal_open_price * (1 - percentage_change) if side == 'Buy' else signal_open_price * (1 + percentage_change)
    
    subsequent_prices = price_data.loc[signal_datetime:]
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            duration = current_datetime - signal_datetime
            return current_datetime, percentage_change_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            duration = current_datetime - signal_datetime
            return current_datetime, percentage_change_price, duration

    return None, None, None

def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None, entry_method=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration', 'Execution Latency'
    ])
    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        if entry_method == 'percentage_change':
            entry_datetime, entry_price, entry_duration = determine_entry_by_percentage(price_data, signal_datetime, percentage_change, side)
            if entry_datetime is None:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Not Filled',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00'
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue
        elif entry_method == 'time_offset':
            entry_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
            if entry_datetime not in price_data.index:
                continue
            entry_price = price_data.at[entry_datetime, 'Open']
            entry_duration = entry_datetime - signal_datetime
        else:
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        
        if duration_str == '00:00:00' and result != 0:
            result = 'Not Filled'
            entry_price = None
            tp_price = None
            sl_price = None
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration)
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data


In [18]:
import os

def generate_report(trade_data, output_directory='E:\Signal Backtesting\Output'):
    def calculate_metrics(group):
        total_trades = len(group)
        total_wins = len(group[group['Result'] == 1])
        total_losses = len(group[group['Result'] == -1])
        win_rate = total_wins / total_trades if total_trades > 0 else 0
        
        roi = ((group['TP Price'].sum() - group['Entry Price'].sum()) / group['Entry Price'].sum()) * 100 if total_trades > 0 else 0
        
        nav = group['Entry Price'].sum()
        drawdown = 0
        cumulative_returns = (group['TP Price'] - group['Entry Price']).cumsum()
        peak = cumulative_returns.cummax()
        drawdown = (peak - cumulative_returns).max()
        
        return {
            'Total Trades': total_trades,
            'Total Wins': total_wins,
            'Total Losses': total_losses,
            'Win Rate': win_rate,
            'ROI': roi,
            'NAV': nav,
            'Max Drawdown': drawdown
        }
    
    report_columns = ['Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    # Calculate metrics for each month
    monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
    monthly_reports = []
    for month, group in monthly_groups:
        metrics = calculate_metrics(group)
        metrics['Period'] = month.strftime('%Y-%m')
        monthly_reports.append(pd.DataFrame([metrics]))

    if monthly_reports:
        report_data = pd.concat(monthly_reports, ignore_index=True)

    # Calculate metrics for the overall period
    overall_metrics = calculate_metrics(trade_data)
    overall_metrics['Period'] = 'Overall'
    overall_report = pd.DataFrame([overall_metrics])
    report_data = pd.concat([report_data, overall_report], ignore_index=True)

    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)
    
    # Save the overall report
    overall_report.to_csv(os.path.join(output_directory, 'overall_report.csv'), index=False)
    
    # Save the monthly reports
    for period in report_data['Period'].unique():
        if period == 'Overall':
            continue
        monthly_report = report_data[report_data['Period'] == period]
        monthly_report.to_csv(os.path.join(output_directory, f'{period}_report.csv'), index=False)

    return report_data

In [19]:
# Load the price data
price_data = pd.read_csv('E:\SignalModel\price 2024-05-01, 2024-06-01 min.csv')
price_data['Datetime'] = pd.to_datetime(price_data['Datetime'])
price_data.set_index('Datetime', inplace=True)
price_data.sort_index(ascending=True, inplace=True)

signal_data = pd.read_csv('E:\SignalModel\price 2024-05-01, 2024-06-01 min.csv')
price_data['Datetime'] = pd.to_datetime(price_data['Datetime'])
price_data.set_index('Datetime', inplace=True)
price_data.sort_index(ascending=True, inplace=True)

In [24]:
result=backtest_trades(price_data, signal_data, tp=0.009, sl=0.016, entry_time_offset=0, percentage_change=0.015, entry_method='percentage_change')
result

,Datetime,Side,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency
0,2024-05-20 21:00:00,Sell,70532.3500,69897.558850,71660.867600,-1,00:11:00,02:17:00
1,2024-05-21 17:00:00,Sell,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00
2,2024-05-21 21:00:00,Buy,68703.6515,69321.984363,67604.393076,-1,04:22:00,40:38:00


In [21]:
report_data = generate_report(result)